In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
rng = np.random.default_rng(67)

def load_and_clean(filename):
    return np.loadtxt(filename, skiprows=2)

data_x = load_and_clean('x24x24.txt')
data_y = load_and_clean('y24x24.txt')
data_z = load_and_clean('z24x24.txt')
full_data = np.vstack([data_x, data_y, data_z])

X = full_data[:, :576]
y = full_data[:, 578]

X_buff, X_test, y_buff, y_test = train_test_split(
    X, y,
    test_size=0.10,
    random_state=67,
    stratify=y   
)

print(len(X_buff))

X_train, X_val, y_train, y_val = train_test_split(
    X_buff, y_buff,
    test_size=0.1,
    random_state=67,
    stratify=y_buff
)

6151


In [2]:
y_train_multiclass = y_train.copy()
y_val_multiclass = y_val.copy()
y_test_multiclass = y_test.copy()
positive_class = 1
y_one = np.where(y_train_multiclass == positive_class, 1, -1)
y_val = np.where(y_val_multiclass == positive_class, 1, -1)
y_test = np.where(y_test_multiclass == positive_class, 1, -1)

In [3]:
def initiateWeights(y):
    sumOnes = np.sum(y==True)
    others = len(y) - sumOnes
    weights = np.where(y==True, 1/sumOnes, 1/others)
    return weights

def normalizeWeights(weights):
    norm_weights = weights/np.sum(weights)
    return norm_weights

In [4]:
weights = initiateWeights(y_one)
weights1 = normalizeWeights(weights)
print(weights1.sum())
weak_learner = DecisionTreeClassifier(max_depth=1)

1.0


training one strong classifier to identify class 1

In [5]:
from sklearn.base import clone
x = 0
amountOfSay = []
stumps_list = []
f = 0
d = 0
iteration = 0
max_iterations = 100
weights = initiateWeights(y_one)
while (f > 0.1 or d < 0.70) and iteration < max_iterations:
    iteration += 1
    weights = normalizeWeights(weights)
    
    new_learner = clone(weak_learner)
    new_learner.fit(X_train, y_one, sample_weight=weights)
    
    stumps_list.append(new_learner)
    
    predictions = stumps_list[-1].predict(X_train)
    
    misclassified = (y_one !=predictions)
    
    epsilon = np.sum(weights[misclassified])/np.sum(weights)
    
    #amount of say
    alpha = 0.5*np.log((1-epsilon)/(epsilon + 1e-10))
    
    #weight actualization
    weights = weights*np.exp(-alpha * y_one * predictions) 
    amountOfSay.append(alpha)
    
    pred = np.zeros(len(X_val))
    
    for a, stump in zip(amountOfSay, stumps_list):
        pred += a*stump.predict(X_val)

    final_results = np.where(pred>=0, 1, -1)
    
    fp = np.sum((y_val== -1) & (final_results == 1))
    tp = np.sum((y_val == 1) & (final_results == 1))
    f = fp/sum(y_val == -1)
    d = tp/sum(y_val==1)
    if(not iteration%5):
        print(iteration)
        print(f)
        print(d) 
    x = x+1
print(f)
print(d)    




5
0.1302170283806344
0.5294117647058824
10
0.13856427378964942
0.6470588235294118
15
0.12687813021702837
0.7058823529411765
0.09682804674457429
0.7058823529411765


predicting on test set

In [6]:
y_pred = np.zeros(len(X_test))
for a, stump in zip(amountOfSay, stumps_list):
    y_pred += a*stump.predict(X_test)    

y_pred = np.where(y_pred>=0, 1, -1)

accuracy measures

In [7]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Value": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average="weighted"),
        recall_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, y_pred, average="weighted")
    ]
})

metrics_df["Value"] = metrics_df["Value"].round(4)
display(metrics_df)

,Metric,Value
0,Accuracy,0.9064
1,Precision,0.9642
2,Recall,0.9064
3,F1,0.9308
